In [1]:
using MajoranaPropagation
using PauliPropagation
using Base.Threads

@show nthreads()

function print_time(seconds)
    hours = div(seconds, 3600)
    minutes = div(seconds % 3600, 60)
    seconds = seconds % 60
    if hours > 0
        return "$(round(Int, hours))h $(round(Int, minutes))m $(round(Int, seconds))s"
    elseif minutes > 0
        return "$(round(Int, minutes))m $(round(Int, seconds))s"
    else
        return "$(seconds)s"
    end
end


SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


nthreads() = 4


print_time (generic function with 1 method)

In [2]:

nx = 3
ny = 3
nspinful = nx * ny
topo = rectangletopology(nx, ny)

U = 8.0
t = 1.0
dt = 0.06

circ_single = []
thetas_single = []

# up hoppings
for (i, j) in topo
    push!(circ_single, FermionicGate(:hopup, [i, j]))
    push!(thetas_single, -t * dt / 2.0)
end

# down hoppings
for (i, j) in topo
    push!(circ_single, FermionicGate(:hopdn, [i, j]))
    push!(thetas_single, -t * dt / 2.0)
end

# on-site repulsion
for i = 1:nspinful
    push!(circ_single, FermionicGate(:nupndn, i))
    push!(thetas_single, U * dt)
end

# down hoppings
for (i, j) in reverse(topo)
    push!(circ_single, FermionicGate(:hopdn, [i, j]))
    push!(thetas_single, -t * dt / 2.0)
end

# up hoppings
for (i, j) in reverse(topo)
    push!(circ_single, FermionicGate(:hopup, [i, j]))
    push!(thetas_single, -t * dt / 2.0)
end

In [3]:

# initial observable
msum = MajoranaSum(nspinful, :nupndn, 3) * MajoranaSum(nspinful, :nupndn, 4) #* MajoranaSum(nspinful, :nupndn, 1) * MajoranaSum(nspinful, :hopup, [5, 6])
#msum = MajoranaSum(nspinful, :nup, 3)
id_val = MajoranaPropagation.pop_id!(msum)

multi_msum = MajoranaSumMulti(msum)
vec_msum = VectorMajoranaSum(msum)
multivec_msum = MultiVectorMajoranaSum(msum)
multivec_msum = MultiVectorMajoranaPropagationCache(multivec_msum)

min_abs_coeff = 1.e-6
max_singles = 8

n_reps = 2

times_multi = zeros(n_reps)
times_vec = zeros(n_reps)
times_multivec = zeros(n_reps)

lengths_multi = zeros(n_reps)
lengths_vec = zeros(n_reps)
lengths_multivec = zeros(n_reps)

for k = 1:n_reps
    println("---$(k)---")

    # normal 
    propagate!(circ_single, msum, thetas_single; min_abs_coeff=min_abs_coeff, max_unpaired=max_singles)

    # vector
    times_vec[k] = @elapsed propagate!(circ_single, vec_msum, thetas_single; min_abs_coeff=min_abs_coeff, max_unpaired=max_singles)
    #println("time vec: $(print_time(times_vec[k]))")

    # multi-vector
    times_multivec[k] = @elapsed propagate!(circ_single, multivec_msum, thetas_single; min_abs_coeff=min_abs_coeff, max_unpaired=max_singles)
    #println("time multivec: $(print_time(times_multivec[k]))")

    @assert length(vec_msum) == length(multivec_msum)
end

---1---
---2---


In [4]:
show_stats(multivec_msum)

length(multivec_msum) / 25

MultiVectorMajoranaPropagationCache stats:
  Weight sector 2: 238 strings (0.26%)
  Weight sector 4: 5745 strings (6.31%)
  Weight sector 6: 25788 strings (28.31%)
  Weight sector 8: 36496 strings (40.07%)
  Weight sector 10: 19014 strings (20.88%)
  Weight sector 12: 3804 strings (4.18%)
  Weight sector 14: 0 strings (0.0%)
Total strings: 91085


3643.4

In [8]:
d = MajoranaPropagation.assign_pools(multivec_msum; max_threads=8)
for k in sort(collect(keys(d)))
    println(k, " => ", d[k])
end

2 => ThreadPools.StaticPool([1])
4 => ThreadPools.StaticPool([8])
6 => ThreadPools.StaticPool([5, 6])
8 => ThreadPools.StaticPool([2, 3, 4])
10 => ThreadPools.StaticPool([7])
12 => ThreadPools.StaticPool([1])
14 => ThreadPools.StaticPool([1])
